# 03 · Multi-head attention
*Partir d_model en heads: formas, qué gana cada una, cuánto cuesta, y las de Gemma*

Notebook del episodio 3 de **Attention is all you need** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/attention-is-all-you-need/multi-head)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** ejemplo a mano: con 1 head dos frases dan lo mismo, con 2 se distinguen → head_i, d_k = 512/8, Concat·W^O → formas (B,T,512)→(B,8,T,64)→(B,T,512) → = nn.MultiheadAttention (3e-7) → 4·d_model² parámetros con 1 u 8 heads → Gemma 2 2B: 8 heads de query, 4 de key/value, head_dim 256, 8·256 ≠ 2304

**Cómo correrlo:** con el entorno de `~/Movies/serie-attention/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "attention"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

**Este episodio usa Gemma 2 2B**, un modelo con licencia de Google. Una sola vez:

1. Aceptá la licencia en https://huggingface.co/google/gemma-2-2b (con tu cuenta de Hugging Face).
2. Creá un token de lectura en https://huggingface.co/settings/tokens.
3. En Colab, ícono de la llave 🔑 (Secretos) → agregá `HF_TOKEN` con tu token y habilitá el acceso del notebook.

El modelo pesa ~5 GB y en float32 usa ~10 GB de RAM: entra en Colab gratis, con poco margen.

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN cargado desde los secretos de Colab")
except Exception as e:
    print("Sin secreto de Colab (" + type(e).__name__ + "). Fuera de Colab: `huggingface-cli login` o definí HF_TOKEN.")

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Attention is all you need. Episodio tres: multi-head attention. En el episodio dos escribimos la atención con una sola matriz de pesos A. El paper no usa una: usa ocho, en paralelo. Hoy, por qué, cómo se parte el vector en heads sin perder nada, cuánto cuesta, y cómo lo hace Gemma.*

## Parte 1 · Un solo reparto no alcanza

Con una sola atención, cada token reparte su peso una sola vez. A veces necesita dos cosas distintas, de dos lugares distintos.

### La idea

- Una head: cada token tiene un único reparto $A$
- Si necesita dos tokens, los promedia: se mezclan
- Multi-head: $h$ atenciones chicas, cada una con sus pesos
- Se concatenan y una matriz $W^O$ las combina

Con una sola head, cada token tiene una única fila de pesos: un único reparto de atención. Head es como se le dice a cada una de estas atenciones, en inglés, en todo el código. Si un token necesita información de dos tokens a la vez, por ejemplo el sujeto y el verbo, lo único que puede hacer es repartir el peso entre los dos, y lo que recibe es el promedio de sus values: mezclado. Multi-head attention hace h atenciones más chicas en paralelo, cada una con sus propias matrices para query, key y value. Las salidas se pegan una al lado de la otra, y una última matriz, W O, las combina.

### El promedio borra información

Un ejemplo hecho a mano, con tres tokens. En la frase A, el value del token cero es uno, cero, y el del uno es cero, uno. La frase B es igual pero con esos dos values al revés. Una sola head que quiere información de los tokens cero y uno: su query se parece igual a las dos keys. Dos heads: la primera busca solo el token cero, la segunda solo el uno. Y las salidas de las dos se concatenan. Con una head, las dos frases dan lo mismo: cero coma cinco, cero coma cinco. El modelo no puede distinguirlas. Con dos heads, la frase A da uno, cero, cero, uno, y la B, cero, uno, uno, cero: distintas. Nada se perdió.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_promedio.py
__file__ = 'snippets/ep03_promedio.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
k = torch.tensor([[1., 0.], [0., 1.], [0., 0.]])  # keys de 3 tokens
def leer(q, V):                                   # una head
    return torch.softmax(q @ k.T, -1) @ V

In [3]:
A = torch.tensor([[1., 0.], [0., 1.], [0., 0.]])  # values: frase A
B = torch.tensor([[0., 1.], [1., 0.], [0., 0.]])  # frase B: 0 y 1 al revés

una = torch.tensor([10., 10.])        # 1 head: quiere los tokens 0 y 1
q1, q2 = torch.tensor([10., 0.]), torch.tensor([0., 10.])  # 2 heads
for nombre, V in [("A", A), ("B", B)]:
    o1 = leer(una, V)
    o2 = torch.cat([leer(q1, V), leer(q2, V)])     # concat de heads
    print(f"frase {nombre}: 1 head {o1.numpy().round(2)}"
          f"   2 heads {o2.numpy().round(2)}")

frase A: 1 head [0.5 0.5]   2 heads [1. 0. 0. 1.]
frase B: 1 head [0.5 0.5]   2 heads [0. 1. 1. 0.]


### Quién mira a quién, en el ejemplo

Los pesos de ese ejemplo. La única head reparte mitad y mitad entre los tokens cero y uno: lo que recibe es el promedio de sus dos values. La head uno pone todo en el token cero. La head dos, todo en el uno. Cada una trae un value entero, sin mezclar, y la concatenación los deja uno al lado del otro.

> **Vaswani et al., 2017 — Attention Is All You Need, sección 3.2.2** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **h = 8 · d_k = 64** · «Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this.»

El paper lo dice en dos frases. Multi-head attention permite atender a la vez a información de distintos subespacios de representación, en distintas posiciones. Con una sola head, el promedio lo impide. Y da los números: h igual a ocho heads, cada una con d sub k igual a d sub v igual a sesenta y cuatro. d sub v es el largo de cada value. Ocho por sesenta y cuatro: quinientos doce, que es d model, el ancho del vector de cada token en el modelo del paper.

## Parte 2 · Las fórmulas y las formas

Ahora la cuenta exacta, y las formas de los tensores en cada paso, que es donde uno se equivoca.

### Multi-head, en fórmulas

$$
\mathrm{head}_i = \mathrm{Attention}\big(X W_i^Q,\; X W_i^K,\; X W_i^V\big)
$$
<small>cada head proyecta $X$ con sus matrices</small>

$$
d_k = d_v = \frac{d_{\text{model}}}{h} = \frac{512}{8} = 64
$$
<small>las 8 heads, juntas, ocupan los 512</small>

$$
\mathrm{MultiHead}(X) = \mathrm{Concat}(\mathrm{head}_1, \dots, \mathrm{head}_8)\, W^O
$$
<small>pegar las salidas y mezclarlas</small>

X es la matriz de entrada: una fila por token, de d model igual a quinientos doce números. Cada head i tiene tres matrices, W i Q, W i K y W i V, de quinientos doce por sesenta y cuatro. En el dibujo, X por W i Q: cada token proyectado a sesenta y cuatro números. Con esas tres proyecciones, la head hace la misma atención del episodio dos. ¿Por qué sesenta y cuatro? Porque es d model dividido por h: en el dibujo, ocho bloques de sesenta y cuatro suman quinientos doce. Concat pega las ocho salidas una al lado de la otra, y vuelve a ser de T por quinientos doce. W O, de quinientos doce por quinientos doce, las mezcla: en el dibujo, cada celda de la salida toma una fila del concat entera, las ocho heads.

### Partir con view, juntar con reshape

En la práctica no hay ocho matrices separadas: hay una grande y se parte. d sub k, quinientos doce dividido ocho. Una sola proyección de quinientos doce a quinientos doce. view reinterpreta las últimas quinientas doce columnas como ocho grupos de sesenta y cuatro, sin copiar nada. transpose pone las heads adelante de los tokens: así cada head queda como una matriz de T por sesenta y cuatro, y el arroba las procesa todas juntas. Los scores: para ver formas usamos Q también como key. Y para juntar, el camino inverso. Uno, cinco, quinientos doce. View: uno, cinco, ocho, sesenta y cuatro. Transpose: uno, ocho, cinco, sesenta y cuatro. Scores: ocho matrices de cinco por cinco. Y el concat vuelve a uno, cinco, quinientos doce.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_formas.py
__file__ = 'snippets/ep03_formas.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn as nn
torch.manual_seed(0)
B, T, d_model, h = 1, 5, 512, 8
X = torch.randn(B, T, d_model)
W_q = nn.Linear(d_model, d_model, bias=False)

In [5]:
d_k = d_model // h                          # 512 / 8 = 64
Q = W_q(X)                                  # [B, T, d_model]
print("Q              ", tuple(Q.shape))
Q = Q.view(B, T, h, d_k)                    # partir las 512 columnas
print("view           ", tuple(Q.shape))
Q = Q.transpose(1, 2)                       # heads adelante
print("transpose(1, 2)", tuple(Q.shape))
S = Q @ Q.transpose(-2, -1) / d_k ** 0.5    # una T×T por head
print("scores         ", tuple(S.shape))
out = S.softmax(-1) @ Q                     # [B, h, T, d_k]
out = out.transpose(1, 2).reshape(B, T, d_model)   # concat
print("concat         ", tuple(out.shape))

Q               (1, 5, 512)
view            (1, 5, 8, 64)
transpose(1, 2) (1, 8, 5, 64)
scores          (1, 8, 5, 5)
concat          (1, 5, 512)


### Las formas, paso a paso (salida real)

El mismo recorrido en cajas. X entra, y una sola matriz lo proyecta a Q, con la misma forma. Partir: view separa las columnas en ocho por sesenta y cuatro, y transpose deja las heads como una dimensión más, como si fueran un batch. Atender: cada head arma su propia tabla de cinco por cinco, y sus salidas son de cinco por sesenta y cuatro. Juntar: transpose y reshape las pegan de vuelta en quinientos doce columnas. Las columnas cero a sesenta y tres son la head uno, las sesenta y cuatro a ciento veintisiete, la head dos, y así.

X: 1×5×512 → Q: 1×5×512 → view: 1×5×8×64 → heads: 1×8×5×64 → scores: 1×8×5×5 → out: 1×8×5×64 → concat: 1×5×512

### Contra nn.MultiheadAttention

Ahora contra PyTorch. nn punto MultiheadAttention, sin sesgos y con el batch primero. Guarda las tres matrices apiladas en una sola, in proj weight, de mil quinientos treinta y seis por quinientos doce; chunk la corta en tres. Nuestra función parte cada proyección en ocho bloques de sesenta y cuatro columnas: X por W Q es lo mismo que pegar X por W uno Q, X por W dos Q, y así. Llamamos a la attention del episodio dos, que ya sabe de batch: las ocho heads van juntas. Y la proyección de salida. La forma: dos, cinco, quinientos doce. La diferencia máxima con PyTorch: tres por diez a la menos siete. Es la misma cuenta.

**Cada línea, su término:**

- `mha.in_proj_weight.chunk(3)` → $W^Q, W^K, W^V \in \mathbb{R}^{512 \times 512}$ — PyTorch las guarda apiladas: 1536 × 512
- `partir = lambda M` → $X W^Q = \big[\, X W_1^Q \mid \cdots \mid X W_8^Q \,\big]$ — la head $i$ son 64 columnas
- `out, _ = attention(Q, K, V)` → $\mathrm{head}_i = \mathrm{softmax}\Big(\frac{Q_i K_i^\top}{\sqrt{64}}\Big) V_i$ — la función del episodio 2
- `@ W_o.T` → $\mathrm{Concat}(\mathrm{head}_1, \dots, \mathrm{head}_8)\, W^O$ — out_proj

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_mha.py
__file__ = 'snippets/ep03_mha.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import torch
import torch.nn as nn
torch.manual_seed(0)


def attention(Q, K, V, mask=None):
    S = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if mask is not None:
        S = S.masked_fill(mask, float("-inf"))
    A = torch.softmax(S, dim=-1)
    return A @ V, A


B, T, d_model, h = 2, 5, 512, 8
X = torch.randn(B, T, d_model)

In [7]:
mha = nn.MultiheadAttention(d_model, h, bias=False, batch_first=True)
W_q, W_k, W_v = mha.in_proj_weight.chunk(3)       # [512, 512] cada una
W_o = mha.out_proj.weight

def multi_head(X):
    d_k = d_model // h
    partir = lambda M: M.view(B, T, h, d_k).transpose(1, 2)
    Q, K, V = partir(X @ W_q.T), partir(X @ W_k.T), partir(X @ W_v.T)
    out, _ = attention(Q, K, V)                     # 8 heads en paralelo
    return out.transpose(1, 2).reshape(B, T, d_model) @ W_o.T

ref, _ = mha(X, X, X, need_weights=False)
print("forma:", tuple(multi_head(X).shape))
print("diferencia máxima:", f"{(multi_head(X) - ref).abs().max():.1e}")

forma: (2, 5, 512)
diferencia máxima: 3.0e-07


## Parte 3 · Ocho heads cuestan lo mismo que una

Ocho atenciones en vez de una suena a ocho veces más trabajo. No lo es.

### Las cuentas del costo

$$
h \cdot (3\, d_{\text{model}}\, d_k) + d_{\text{model}}^2 = 4\, d_{\text{model}}^2
$$
<small>parámetros: no dependen de $h$, porque $h\, d_k = d_{\text{model}}$</small>

$$
h \cdot T^2 d_k = T^2\, d_{\text{model}}
$$
<small>multiplicaciones de $QK^\top$: tampoco</small>

Los parámetros. Cada head tiene tres matrices de d model por d sub k, y W O es de d model por d model. Como h por d sub k es d model, el total es cuatro veces d model al cuadrado, haya una head u ocho. Con quinientos doce, un millón cuarenta y ocho mil quinientos setenta y seis. Las cuentas. Una head con d sub k igual a quinientos doce hace, por cada celda de la tabla de T por T, quinientos doce productos. Ocho heads hacen ocho tablas, pero cada celda solo cuesta sesenta y cuatro: en el dibujo, la fila de Q sub i tiene sesenta y cuatro números. Ocho por sesenta y cuatro, quinientos doce. Lo mismo.

### Contarlo

Armamos nn punto MultiheadAttention con una head y con ocho. Contamos parámetros con numel, que da la cantidad de números de cada tensor. Y las multiplicaciones de la atención para cien tokens: Q por K transpuesta y A por V, en todas las heads. Una head: un millón cuarenta y ocho mil quinientos setenta y seis parámetros, y diez millones doscientas cuarenta mil multiplicaciones. Ocho heads: exactamente lo mismo. El paper lo dice así: por la dimensión reducida de cada head, el costo total es parecido al de una sola head con la dimensión completa. Lo que cambia es qué se puede expresar.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_costo.py
__file__ = 'snippets/ep03_costo.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch.nn as nn
d_model, T = 512, 100

In [9]:
for h in [1, 8]:
    mha = nn.MultiheadAttention(d_model, h, bias=False)
    params = sum(p.numel() for p in mha.parameters())
    d_k = d_model // h
    mults = h * (T * T * d_k) * 2      # Q·Kᵀ y A·V, todas las heads
    print(f"h={h}  d_k={d_k:3}  parámetros={params:,}"
          f"  multiplicaciones (T=100)={mults:,}")

h=1  d_k=512  parámetros=1,048,576  multiplicaciones (T=100)=10,240,000
h=8  d_k= 64  parámetros=1,048,576  multiplicaciones (T=100)=10,240,000


### Qué gana cada head

- Su propio reparto: $h$ filas de pesos por token, no una
- Su propio subespacio: compara 64 coordenadas proyectadas
- En modelos entrenados, algunas heads se especializan
- Una head sola rinde peor: lo medimos en el ep. 8

Entonces, qué gana. Cada token tiene ahora ocho repartos de atención, no uno: puede mirar al sujeto con una head y al verbo con otra. Cada head compara queries y keys en su propio espacio de sesenta y cuatro coordenadas, proyectadas con sus propias matrices: lo que una considera parecido, otra no. En modelos entrenados, algunas heads hacen tareas reconocibles, como mirar al token anterior o copiar un nombre: en Alineación, episodio siete, vimos un circuito de veintiséis attention heads en GPT dos small. Ojo: no todas tienen una función clara. Y que una sola head rinde peor que ocho, el paper lo mide en su tabla tres; la leemos en el episodio ocho.

## Parte 4 · Las heads de Gemma

Siete años después, Gemma dos sigue usando multi-head attention, con un cambio: no todas las heads tienen su propia key.

### Leer la configuración y las formas

La configuración de Gemma dos, dos B, sin cargar el modelo. Y las formas de las cuatro matrices de atención de la capa cero, leídas del archivo. Y una cuenta de juguete: cuatro heads de key, repetidas dos veces cada una. Ocho heads de query, cuatro heads de key y value, de doscientos cincuenta y seis cada una. q proj: dos mil cuarenta y ocho por dos mil trescientos cuatro. k proj y v proj, la mitad: mil veinticuatro. o proj vuelve a dos mil trescientos cuatro. Y K pasa de cuatro heads a ocho. Esto es GQA, grouped-query attention: dos heads de query comparten cada key y cada value.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_gemma.py
__file__ = 'snippets/ep03_gemma.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import torch
from transformers import AutoConfig
from huggingface_hub import hf_hub_download
from safetensors import safe_open
REPO = "google/gemma-2-2b"
mapa = json.load(open(hf_hub_download(REPO, "model.safetensors.index.json")))["weight_map"]

In [11]:
cfg = AutoConfig.from_pretrained(REPO)
print("heads de query:", cfg.num_attention_heads,
      "· heads de key/value:", cfg.num_key_value_heads,
      "· head_dim:", cfg.head_dim)
for p in ["q_proj", "k_proj", "v_proj", "o_proj"]:
    n = f"model.layers.0.self_attn.{p}.weight"
    with safe_open(hf_hub_download(REPO, mapa[n]), framework="pt") as f:
        print(f"{p}: {f.get_slice(n).get_shape()}")
K = torch.randn(5, 4, 256)                    # T=5, 4 heads de key
K8 = K.repeat_interleave(2, dim=1)            # cada una, para 2 queries
print("K", tuple(K.shape), "→", tuple(K8.shape))

heads de query: 8 · heads de key/value: 4 · head_dim: 256
q_proj: [2048, 2304]
k_proj: [1024, 2304]
v_proj: [1024, 2304]
o_proj: [2304, 2048]
K (5, 4, 256) → (5, 8, 256)


### GQA: dos queries por cada key

$$
\mathrm{head}_i = \mathrm{Attention}\big(Q_i,\; K_{\lceil i/2 \rceil},\; V_{\lceil i/2 \rceil}\big)
$$
<small>las heads 1 y 2 usan la key 1; la 3 y la 4, la 2…</small>

$$
8 \times 256 = 2048 \neq 2304 = d_{\text{model}}
$$
<small>en Gemma, $h\, d_k$ no es $d_{\text{model}}$</small>

En fórmula: la head de query i usa la key y el value número i sobre dos, redondeado hacia arriba. En el dibujo, cada una de las cuatro keys se copia dos veces y quedan ocho, una por head de query. ¿Para qué? Al generar texto, las keys y values de los tokens anteriores se guardan en memoria para no recalcularlos, y con la mitad de heads ocupan la mitad. Y un detalle que rompe la regla del paper: ocho heads de doscientos cincuenta y seis dan dos mil cuarenta y ocho, no dos mil trescientos cuatro. d sub k ya no es d model dividido h. No pasa nada: W O, en el dibujo, lleva las dos mil cuarenta y ocho columnas de vuelta a dos mil trescientos cuatro. Lo vemos entero en el episodio nueve.

> **Gemma Team, 2024 — Gemma 2: Improving Open Language Models at a Practical Size** · [arXiv:2408.00118](https://arxiv.org/abs/2408.00118)
>
> **8 heads · 4 KV heads** · Tabla 1, modelo 2B: d_model 2304, «Num heads» 8, «Num KV heads» 4, «Head size» 256, «Head type» GQA. «We use GQA with num_groups = 2».

El reporte técnico de Gemma dos lo confirma. En su tabla uno, para el modelo de dos B: d model, dos mil trescientos cuatro; ocho heads; cuatro heads de key y value; head size, doscientos cincuenta y seis; y tipo de head, GQA. En el texto: usan GQA con dos grupos. Son los mismos números que leímos del archivo.

## Resumiendo

- Una head = un reparto por token; dos tokens se promedian
- $\mathrm{head}_i = \mathrm{Attention}(XW_i^Q, XW_i^K, XW_i^V)$, luego Concat · $W^O$
- Formas: view → (B, h, T, d_k) con transpose; reshape para juntar
- Mismo costo: $4\,d_{\text{model}}^2$ parámetros con 1 u 8 heads
- Igual a nn.MultiheadAttention (dif. 3e-7)
- Próximo: positional encoding con senos y cosenos